In [1]:
import os
import warnings
import importlib

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys
sys.path.append('/kaggle/input/datasets/keithmarange/cnn-methods/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=RuntimeWarning)

2026-04-11 05:22:07.726107: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775884928.257096      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775884928.375782      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775884929.470823      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775884929.470876      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775884929.470879      24 computation_placer.cc:177] computation placer alr

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_run_name = 'cnn_1d_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
random_search = False

model_target = 'gesture_action'

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.4,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

Train: 1944 seqs | 38.0%
Test:  648 seqs  | 12.7%


In [5]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

cnn_clf = utils.Keras1DCNNClassifier(verbose=0)
custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

pipeline = Pipeline([
    (pipe_name, custom_extractor),
    ('preprocessor', preprocessor),
    ('pca', utils.IndexPreservingPCA()),
    ('classifier', utils.ManyToOneWrapper(
        estimator=cnn_clf,
        extractor=custom_extractor,
        mode=None,
        target=model_target
    ))
])

cv = GroupKFold(n_splits=n_splits)

In [6]:
if random_search:
    param_grid = {
    f'{pipe_name}__imu_sensor_list':        [acc_columns],
    f'{pipe_name}__imu_domain':             ['displacement'],
    f'{pipe_name}__combine_imu_axes':       [False],
    f'{pipe_name}__sampling_rate':          [100],


    f'{pipe_name}__rotation_sensor_list':   [rot_columns],
    f'{pipe_name}__combine_rot_axes':       [False],
    f'{pipe_name}__rotation_domain':        ['frequency'],

    f'{pipe_name}__thermopile_sensor_list': [thm_columns],
    f'{pipe_name}__thermopile_mode':        ['baseline'],
    f'{pipe_name}__tof_sensor_list':        [tof_columns],
    f'{pipe_name}__tof_mode':               ['baseline'],

    f'{pipe_name}__window':                 [0.2, 2],
    f'{pipe_name}__step_sec':               [0.1],

    f'{pipe_name}__dc_offset':              [0],
    f'{pipe_name}__band_edges':             [linear_edges],
    f'{pipe_name}__category_data':          [False],
    f'{pipe_name}__segmentation':           ['window'],

    'pca__n_components': [None],

    'classifier__estimator__filters': [48],      
    'classifier__estimator__kernel_size': [7, 9, 11],            
    'classifier__estimator__dropout': [0.3],
    'classifier__estimator__learning_rate': [0.001],
    'classifier__estimator__epochs': [50],
    'classifier__estimator__batch_size': [16,],
    }
else:
    param_grid = {
    f'{pipe_name}__imu_sensor_list':        [acc_columns],
    f'{pipe_name}__imu_domain':             ['displacement'],
    f'{pipe_name}__combine_imu_axes':       [False],
    f'{pipe_name}__sampling_rate':          [100],


    f'{pipe_name}__rotation_sensor_list':   [rot_columns],
    f'{pipe_name}__combine_rot_axes':       [False],
    f'{pipe_name}__rotation_domain':        ['frequency'],

    f'{pipe_name}__thermopile_sensor_list': [thm_columns],
    f'{pipe_name}__thermopile_mode':        ['baseline'],
    f'{pipe_name}__tof_sensor_list':        [tof_columns],
    f'{pipe_name}__tof_mode':               ['baseline'],

    f'{pipe_name}__window':                 [0.2],
    f'{pipe_name}__step_sec':               [0.1],

    f'{pipe_name}__dc_offset':              [0],
    f'{pipe_name}__band_edges':             [linear_edges],
    f'{pipe_name}__category_data':          [False],
    f'{pipe_name}__segmentation':           ['window'],

    'pca__n_components': [None],

    'classifier__estimator__filters': [48],      
    'classifier__estimator__kernel_size': [7],         
    'classifier__estimator__dropout': [0.3],  
    'classifier__estimator__learning_rate': [0.001],
    'classifier__estimator__epochs': [100],
    'classifier__estimator__batch_size': [16],
}

In [7]:
cv_results_list = []

for col in orientation_cols_dict:
    if random_search:
        search_obj = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grid,
            n_iter=5,
            cv=cv,
            random_state=42,
            n_jobs=1,
            verbose=2,
            return_train_score=True
        )
    else:
        search_obj = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=cv,
            verbose=2,
            n_jobs=1,
            return_train_score=True
        )

    sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
    y = sliced_data_df[['sequence_id', model_target]]

    n_fits = n_splits * len(ParameterGrid(param_grid))

    with utils.tqdm_joblib(tqdm(total=n_fits, desc=f"CV {col}")):
        search_obj.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

    if save_model:
        model = search_obj.best_estimator_
        path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
        joblib.dump(model, path_to_model_run_name)

    cv_results_df = pd.DataFrame(search_obj.cv_results_)
    cv_results_df['orientation_data'] = col
    cv_results_list.append(cv_results_df)

master_cv_results_df = pd.concat(cv_results_list)
master_cv_results_df['model_target'] = model_target
path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
master_cv_results_df.to_csv(path_to_cv_results, index=False)

CV Lie on Back:   0%|          | 0/3 [00:00<?, ?it/s]

Fitting 3 folds for each of 1 candidates, totalling 3 fits


I0000 00:00:1775885025.518895      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775885025.521501      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1775885029.564018      74 service.cc:152] XLA service 0x345b8b40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775885029.564062      74 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775885029.564068      74 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775885030.142050      74 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775885033.818030      74 device_compiler.h:188] Compiled cluster u

[CV] END classifier__estimator__batch_size=16, classifier__estimator__dropout=0.3, classifier__estimator__epochs=100, classifier__estimator__filters=48, classifier__estimator__kernel_size=7, classifier__estimator__learning_rate=0.001, imu_extractor__band_edges=[ 0 10 20 30 40 50], imu_extractor__category_data=False, imu_extractor__combine_imu_axes=False, imu_extractor__combine_rot_axes=False, imu_extractor__dc_offset=0, imu_extractor__imu_domain=displacement, imu_extractor__imu_sensor_list=['acc_x', 'acc_y', 'acc_z'], imu_extractor__rotation_domain=frequency, imu_extractor__rotation_sensor_list=['rot_w', 'rot_x', 'rot_y', 'rot_z'], imu_extractor__sampling_rate=100, imu_extractor__segmentation=window, imu_extractor__step_sec=0.1, imu_extractor__thermopile_mode=baseline, imu_extractor__thermopile_sensor_list=['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5'], imu_extractor__tof_mode=baseline, imu_extractor__tof_sensor_list=['tof_1_v0', 'tof_1_v1', 'tof_1_v2', 'tof_1_v3', 'tof_1_v4', 'tof_1_v5

In [8]:
if do_report:

    best_model = search_obj.best_estimator_

    extractor    = best_model.named_steps['imu_extractor']
    preprocessor = best_model.named_steps['preprocessor']
    classifier   = best_model.named_steps['classifier']

    X_feat = extractor.transform(test_sample_df)
    X_proc = preprocessor.transform(X_feat)

    y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
    y_true = y_true.reindex(X_proc.index)

    y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

    print(classification_report(y_true, y_pred))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).T.sort_values('f1-score', ascending=False)

    report_df.to_csv(model_run_folder_name + 'cnn_1d_v1_per_gesture_scores.csv')